# Importing Libraries

In [28]:
from langchain_community.llms import Ollama
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict

In [29]:

telugu_llm = Ollama(model="gemma:2b")
english_llm = Ollama(model="gemma:2b")
translator_llm = Ollama(model="qwen2.5:1.5b")

# Defining States

In [30]:
class PipelineState(TypedDict):
    input_telugu: str
    telugu_summary: str
    english_text: str
    english_summary: str

In [31]:
class TranslationState(TypedDict):
    input_text: str
    translated_text: str

In [32]:
def te_to_en_translate(state: TranslationState):
    prompt = f"""
Translate the following Telugu text into clear, natural English.
Do not summarize. Do not add information.

Text:
{state["input_text"]}
"""
    return {
        "translated_text": translator_llm.invoke(prompt)
    }


# Defining Sub Graph

In [33]:
from langgraph.graph import StateGraph, START, END

translation_graph = StateGraph(TranslationState)
translation_graph.add_node("translate", te_to_en_translate)
translation_graph.add_edge(START, "translate")
translation_graph.add_edge("translate", END)

translation_subgraph = translation_graph.compile()


# Nodes

In [34]:
def summarize_telugu(state: PipelineState):
    prompt = f"""
క్రింది తెలుగులో ఉన్న పాఠ్యాన్ని సంక్షిప్తంగా, ముఖ్యాంశాలు మాత్రమే ఉంచుతూ సంగ్రహించండి.

పాఠ్యం:
{state["input_telugu"]}
"""
    return {
        "telugu_summary": telugu_llm.invoke(prompt)
    }


In [ ]:
def translate_telugu_summary(state: PipelineState):
    result = translation_subgraph.invoke({
        "input_text": state["telugu_summary"]   # callng the subgraph
    })
    return {
        "english_text": result["translated_text"]
    }


In [36]:
def summarize_english(state: PipelineState):
    prompt = f"""
Summarize the following English text clearly and concisely.
Preserve key facts.

Text:
{state["english_text"]}
"""
    return {
        "english_summary": english_llm.invoke(prompt)
    }


# Define Graph

In [37]:
pipeline = StateGraph(PipelineState)

pipeline.add_node("summarize_telugu", summarize_telugu)
pipeline.add_node("translate_te_en", translate_telugu_summary)
pipeline.add_node("summarize_english", summarize_english)

pipeline.add_edge(START, "summarize_telugu")
pipeline.add_edge("summarize_telugu", "translate_te_en")
pipeline.add_edge("translate_te_en", "summarize_english")
pipeline.add_edge("summarize_english", END)

graph = pipeline.compile()


# Input and Output

In [38]:
telugu_text = """
పాకిస్తాన్ విమానయాన రంగంలో భారీ వివాదం చెలరేగింది.
262 మంది పైలట్లు నకిలీ లైసెన్సులు కలిగి ఉన్నట్లు ఆరోపణలు వచ్చాయి.
ఈ ఘటన ప్రపంచవ్యాప్తంగా విమాన భద్రతపై ఆందోళనను రేకెత్తించింది.
"""

result = graph.invoke({
    "input_telugu": telugu_text
})

print("TELUGU SUMMARY:\n", result["telugu_summary"])
print("\nFINAL ENGLISH SUMMARY:\n", result["english_summary"])


TELUGU SUMMARY:
 **Summary**

The passage describes a train accident that occurred in a railway station.

**Key Points**

* 262 people were trapped on a train.
* They were all passengers on board the train.
* The train was involved in a collision.
* The accident caused a lot of damage and panic.
* The emergency services were quickly called and responded to the incident.

FINAL ENGLISH SUMMARY:
 Sure, here's a summary of the text in a clear and concise manner:

A train crash trapped 262 people on board and caused significant damage and panic.
